<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/enzo%2Fp10-8-clinical-expansion-preflight/72_P10_8_AlKafri_mask_semantics_and_label_normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 72 — P10.8: semántica de máscaras y normalización de etiquetas Al‑Kafri

Audita máscaras manuales/procesadas, T1/T2, tokens D3/D4/D5, colores PNG, pairing, XCF y documentación. **No entrena, no carga `.pt`, no abre tests sellados y no crea ground truth clínico.** D3/D4/D5 son tokens opacos hasta que una fuente explícita demuestre su significado.


In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Entorno no Colab')


Mounted at /content/drive


In [3]:
import os

os.environ["PFI_ALKAFRI_ROOT"] = (
    "/content/drive/MyDrive/PFI_MVP/data/"
    "AXIAL_ALKAFRI/extracted/_nested"
)

os.environ["PFI_P10_8_NOTEBOOK71_ROOT"] = (
    "/content/drive/MyDrive/PFI_MVP/results/"
    "P10_8_clinical_expansion_preflight/"
    "sudirman_alkafri_audit_v2"
)

print("Variables configuradas")

Variables configuradas


In [4]:
from __future__ import annotations
import hashlib,json,os,re,shutil
from collections import defaultdict
from datetime import datetime,timezone
from pathlib import Path
import numpy as np,pandas as pd
from PIL import Image

ROOT=Path(os.getenv('PFI_ROOT','/content/drive/MyDrive/PFI_MVP'))
PREF=Path(os.getenv('PFI_P10_8_PREFLIGHT_ROOT',str(ROOT/'results/P10_8_clinical_expansion_preflight')))
N71=Path(os.getenv('PFI_P10_8_NOTEBOOK71_ROOT',str(PREF/'sudirman_alkafri_audit_v2')))
m71=N71/'NOTEBOOK_71_COMPLETE.json'
if not m71.is_file(): raise FileNotFoundError(f'Falta marcador 71: {m71}')
j71=json.loads(m71.read_text(encoding='utf-8'))
if j71.get('status')!='NOTEBOOK_71_COMPLETE' or j71.get('trainingExecuted') is not False or j71.get('weightsDeserialized') is not False: raise RuntimeError('Notebook 71 inválido')
if j71.get('inventoryTruncated') is not False or int(j71.get('indexedFileCount',0))<1000: raise RuntimeError('Notebook 71 no auditó fuente primaria completa')

ALK=Path(os.getenv('PFI_ALKAFRI_ROOT',str(ROOT/'data/AXIAL_ALKAFRI/extracted/_nested')))
if not ALK.is_dir(): raise FileNotFoundError(ALK)
BASE=ALK.parent.parent if ALK.name=='_nested' else ALK
MAN=ALK/'ground_truth__Manual_Label_Data/03_Manual_Label_Data'
PROC=ALK/'ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data'
MRI=ALK/'main_dataset__MRI_Data/01_MRI_Data'
E7=ROOT/'results/E7_alkafri_axial_curated_subset'
for p in [MAN,PROC,MRI]:
    if not p.is_dir(): raise FileNotFoundError(p)
OUT=Path(os.getenv('PFI_P10_8_NOTEBOOK72_ROOT',str(PREF/'alkafri_mask_semantics_and_label_normalization')))
NS=int(os.getenv('PFI_P10_8_MASK_SAMPLE_PER_GROUP','12'))
NP=int(os.getenv('PFI_P10_8_MAX_PAIR_COMPARISONS','120'))
NX=int(os.getenv('PFI_P10_8_MAX_XCF_SAMPLES','60'))
ND=int(os.getenv('PFI_P10_8_MAX_DOCUMENT_FILES','600'))
NB=int(os.getenv('PFI_P10_8_MAX_DOCUMENT_BYTES',str(4*1024*1024)))
NR=int(os.getenv('PFI_P10_8_MAX_TABULAR_ROWS','5000'))
sha=lambda s:hashlib.sha256(str(s).encode()).hexdigest()
def write_json(p,x):
    p.parent.mkdir(parents=True,exist_ok=True); t=p.with_suffix(p.suffix+'.tmp')
    t.write_text(json.dumps(x,indent=2,ensure_ascii=False,sort_keys=True)+'\n',encoding='utf-8'); os.replace(t,p)
rx=re.compile(r'^(T[12])_(\d{4})_(D\d+)(?:[^.]*)\.(png|xcf)$',re.I)
def parse(p):
    m=rx.match(p.name)
    return (m.group(1).upper(),sha('alkafri|'+m.group(2)),m.group(3).upper()) if m else (None,None,None)
def source(p):
    s=str(p).lower()
    return 'manual' if 'manual_label_data' in s else ('processed_intermediary' if 'ground_truth_label' in s else 'unknown')

rows=[]
for root in [MAN,PROC]:
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.png','.xcf'}:
            mod,case,d=parse(p)
            rows.append({'sourceType':source(p),'relativePathHash':sha(p.relative_to(ALK)),'suffix':p.suffix.lower(),'sizeBytes':p.stat().st_size,'modality':mod,'caseKeyHash':case,'dToken':d,'nameParsed':case is not None,'_path':p})
priv=pd.DataFrame(rows)
if priv.empty: raise RuntimeError('No se encontraron PNG/XCF')
public=priv.drop(columns=['_path'])
summary=public.groupby(['sourceType','suffix','modality','dToken','nameParsed'],dropna=False).agg(fileCount=('relativePathHash','count'),totalBytes=('sizeBytes','sum'),uniqueCaseCount=('caseKeyHash','nunique')).reset_index()
print('Máscaras/XCF:',len(priv),'PNG:',int((priv.suffix=='.png').sum()),'XCF:',int((priv.suffix=='.xcf').sum()))
display(summary)

png=priv[priv.suffix=='.png'].copy()
samples=pd.concat([g.sort_values('relativePathHash').head(NS) for _,g in png.groupby(['sourceType','modality','dToken'],dropna=False)],ignore_index=True)
def hx(c): return '#'+''.join(f'{int(v):02X}' for v in c.tolist()[:4])
profiles=[]
for _,r in samples.iterrows():
    try:
        with Image.open(r._path) as im:
            mode=im.mode; w,h=im.size; a=np.asarray(im.convert('RGBA'),dtype=np.uint8)
        colors,counts=np.unique(a.reshape(-1,4),axis=0,return_counts=True); order=np.argsort(counts)[::-1]; colors,counts=colors[order],counts[order]
        n=len(colors); total=a.shape[0]*a.shape[1]
        profiles.append({'sourceType':r.sourceType,'relativePathHash':r.relativePathHash,'modality':r.modality,'caseKeyHash':r.caseKeyHash,'dToken':r.dToken,'width':w,'height':h,'originalMode':mode,'uniqueColorCount':n,'maskProfile':'SINGLE_COLOR' if n==1 else ('BINARY_COLOR' if n==2 else 'MULTICOLOR'),'dominantColorHex':hx(colors[0]),'dominantColorFraction':float(counts[0]/total),'secondColorHex':hx(colors[1]) if n>1 else None,'secondColorFraction':float(counts[1]/total) if n>1 else None,'colorSignatureSha256':sha('|'.join(f'{hx(c)}:{int(k)}' for c,k in zip(colors,counts))),'readSucceeded':True,'readErrorType':None})
    except Exception as e:
        profiles.append({'sourceType':r.sourceType,'relativePathHash':r.relativePathHash,'modality':r.modality,'caseKeyHash':r.caseKeyHash,'dToken':r.dToken,'width':None,'height':None,'originalMode':None,'uniqueColorCount':None,'maskProfile':'UNREADABLE','dominantColorHex':None,'dominantColorFraction':None,'secondColorHex':None,'secondColorFraction':None,'colorSignatureSha256':None,'readSucceeded':False,'readErrorType':type(e).__name__})
profile=pd.DataFrame(profiles)
print('PNG perfilados:',len(profile)); display(profile.groupby(['sourceType','modality','dToken','maskProfile'],dropna=False).size().reset_index(name='sampleCount'))

pairable=png[png.nameParsed & png.caseKeyHash.notna() & png.modality.notna() & png.dToken.notna()]
keys=['caseKeyHash','modality','dToken']
mg={k:g.sort_values('relativePathHash') for k,g in pairable[pairable.sourceType=='manual'].groupby(keys)}
pg={k:g.sort_values('relativePathHash') for k,g in pairable[pairable.sourceType=='processed_intermediary'].groupby(keys)}
pairs=[]
for k in sorted(set(mg)|set(pg)):
    a,b=len(mg.get(k,[])),len(pg.get(k,[]))
    st='ONE_TO_ONE' if a==b==1 else ('ONE_OR_MANY_TO_ONE_OR_MANY' if a and b else ('MANUAL_ONLY' if a else 'PROCESSED_ONLY'))
    pairs.append({'caseKeyHash':k[0],'modality':k[1],'dToken':k[2],'manualFileCount':a,'processedFileCount':b,'pairStatus':st})
pairreg=pd.DataFrame(pairs)
cand=pairreg[(pairreg.manualFileCount>0)&(pairreg.processedFileCount>0)].copy()
cand['sortKey']=(cand.caseKeyHash+'|'+cand.modality+'|'+cand.dToken).map(sha)
comp=[]
for _,r in cand.sort_values('sortKey').head(NP).iterrows():
    k=(r.caseKeyHash,r.modality,r.dToken); ma=mg[k].iloc[0]; pr=pg[k].iloc[0]
    x={'caseKeyHash':k[0],'modality':k[1],'dToken':k[2],'manualRelativePathHash':ma.relativePathHash,'processedRelativePathHash':pr.relativePathHash,'shapeMatch':False,'exactPixelMatch':False,'manualColorCount':None,'processedColorCount':None,'colorSetJaccard':None,'comparisonSucceeded':False,'comparisonErrorType':None}
    try:
        with Image.open(ma._path) as im: A=np.asarray(im.convert('RGBA'),dtype=np.uint8)
        with Image.open(pr._path) as im: B=np.asarray(im.convert('RGBA'),dtype=np.uint8)
        x['shapeMatch']=A.shape==B.shape; x['exactPixelMatch']=bool(x['shapeMatch'] and np.array_equal(A,B))
        ca={tuple(z) for z in np.unique(A.reshape(-1,4),axis=0).tolist()}; cb={tuple(z) for z in np.unique(B.reshape(-1,4),axis=0).tolist()}; u=ca|cb
        x.update(manualColorCount=len(ca),processedColorCount=len(cb),colorSetJaccard=float(len(ca&cb)/len(u)) if u else 1.0,comparisonSucceeded=True)
    except Exception as e: x['comparisonErrorType']=type(e).__name__
    comp.append(x)
comparison=pd.DataFrame(comp)
print('Claves pairing:',len(pairreg),'comparaciones:',len(comparison))

xcf=priv[priv.suffix=='.xcf'].sort_values('relativePathHash').head(NX)
xr=[]
for _,r in xcf.iterrows():
    try:
        head=r._path.open('rb').read(32); valid=head.startswith(b'gimp xcf')
        xr.append({'relativePathHash':r.relativePathHash,'modality':r.modality,'caseKeyHash':r.caseKeyHash,'dToken':r.dToken,'sizeBytes':r.sizeBytes,'xcfHeaderValid':valid,'externalLayerParserAvailable':bool(shutil.which('xcfinfo') or shutil.which('gimp') or shutil.which('gimp-console')),'layerNamesExtracted':False,'layerCount':None,'auditStatus':'VALID_XCF_HEADER_LAYER_SEMANTICS_NOT_PARSED' if valid else 'INVALID_OR_UNKNOWN_XCF_HEADER'})
    except Exception as e:
        xr.append({'relativePathHash':r.relativePathHash,'modality':r.modality,'caseKeyHash':r.caseKeyHash,'dToken':r.dToken,'sizeBytes':r.sizeBytes,'xcfHeaderValid':False,'externalLayerParserAvailable':False,'layerNamesExtracted':False,'layerCount':None,'auditStatus':'XCF_READ_FAILED_'+type(e).__name__})
xcfa=pd.DataFrame(xr)

suffixes={'.m','.txt','.md','.csv','.tsv','.json','.xml','.yaml','.yml'}
patterns={'D3':r'\bD3\b','D4':r'\bD4\b','D5':r'\bD5\b','disc':r'\b(disc|disk|intervertebral)\b','vertebra':r'\bvertebr','thecal_sac':r'thecal\s+sac|dural\s+sac','facet':r'facet|zygapophy|facetar','ligamentum_flavum':r'ligamentum\s+flavum|ligamento\s+amarillo|flavum','nerve_root':r'nerve\s+root|ra[ií]z\s+nerv|radicular','epidural_fat':r'epidural\s+fat|grasa\s+epidural','annular_tear':r'annular\s+(tear|fissure)|(desgarro|fisura)\s+anular','herniation':r'herniation|hernia\s+discal','bulging':r'bulging|disc\s+bulge|abombamiento','disc_height':r'(disc|disk)\s+height|altura\s+discal','spondylolisthesis':r'spondylolisthesis|anterolisthesis|retrolisthesis|listesis'}
patterns={k:re.compile(v,re.I) for k,v in patterns.items()}
docs=[]
for alias,root in [('dataset_base',BASE),('e7_results',E7)]:
    if root.exists():
        for p in root.rglob('*'):
            if p.is_file() and p.suffix.lower() in suffixes and p.stat().st_size<=NB:
                docs.append((sha(str(p)),alias,p))
docs=sorted(docs)[:ND]; dr=[]; source_terms=defaultdict(set)
for _,alias,p in docs:
    rid=sha(f'{alias}|{p.name}|{p.stat().st_size}'); text=''; status='READ_OK'
    try:
        if p.suffix.lower() in {'.csv','.tsv'}:
            df=pd.read_csv(p,sep='\t' if p.suffix.lower()=='.tsv' else ',',nrows=NR,dtype=str,low_memory=False)
            text=' '.join(map(str,df.columns))+' '+' '.join(df.fillna('').astype(str).to_numpy().ravel().tolist())
        else: text=p.read_text(encoding='utf-8',errors='replace')
    except Exception as e: status='READ_FAILED_'+type(e).__name__
    terms=sorted(k for k,v in patterns.items() if text and v.search(text)); source_terms[rid].update(terms)
    dr.append({'rootAlias':alias,'relativePathHash':rid,'suffix':p.suffix.lower(),'sizeBytes':p.stat().st_size,'readStatus':status,'matchedTerms':'|'.join(terms),'matchedTermCount':len(terms)})
docaudit=pd.DataFrame(dr)

sem=[]
for d in ['D3','D4','D5']:
    co=set()
    for terms in source_terms.values():
        if d in terms: co.update(t for t in terms if t not in {'D3','D4','D5'})
    z=public[public.dToken==d]
    sem.append({'dToken':d,'manualPngCount':int(((z.sourceType=='manual')&(z.suffix=='.png')).sum()),'processedPngCount':int(((z.sourceType=='processed_intermediary')&(z.suffix=='.png')).sum()),'xcfCount':int((z.suffix=='.xcf').sum()),'cooccurringDocumentationTerms':'|'.join(sorted(co)),'documentedExactMappingFound':False,'semanticStatus':'SEMANTICS_INFERRED_REQUIRES_REVIEW' if co else 'UNKNOWN_NOT_USABLE','anatomicalMeaningValidated':False,'clinicalFindingMeaningValidated':False,'trainingAuthorized':False})
semantics=pd.DataFrame(sem)
fmap={'facet_hypertrophy':'facet','ligamentum_flavum_hypertrophy':'ligamentum_flavum','annular_tear':'annular_tear','nerve_root_compression':'nerve_root','epidural_fat':'epidural_fat','disc_height':'disc_height','spondylolisthesis':'spondylolisthesis','disc_herniation':'herniation','disc_bulging':'bulging'}
allterms={t for v in source_terms.values() for t in v}
support=pd.DataFrame([{'findingType':f,'documentationTermPresent':t in allterms,'maskClassMappingValidated':False,'caseSeriesLevelSliceAlignmentValidated':False,'clinicalTaxonomyFrozen':False,'supportStatus':'DOCUMENTATION_TERM_PRESENT_MASK_MAPPING_UNVALIDATED' if t in allterms else 'ANNOTATION_NOT_DEMONSTRATED','trainingAuthorized':False} for f,t in fmap.items()])
display(semantics); display(support)

OUT.mkdir(parents=True,exist_ok=True)
paths={'maskInventory':OUT/'mask_file_inventory_v1.csv','inventorySummary':OUT/'mask_inventory_summary_v1.csv','maskProfile':OUT/'sampled_png_mask_profile_v1.csv','pairingRegistry':OUT/'manual_processed_pairing_registry_v1.csv','pairComparison':OUT/'manual_processed_pixel_comparison_v1.csv','xcfAudit':OUT/'xcf_header_and_layer_audit_v1.csv','documentAudit':OUT/'documentation_and_code_term_audit_v1.csv','semanticsRegistry':OUT/'d_token_semantics_registry_v1.csv','candidateSupport':OUT/'candidate_finding_support_matrix_v1.csv','summaryJson':OUT/'NOTEBOOK_72_SUMMARY.json'}
for df,k in [(public,'maskInventory'),(summary,'inventorySummary'),(profile,'maskProfile'),(pairreg,'pairingRegistry'),(comparison,'pairComparison'),(xcfa,'xcfAudit'),(docaudit,'documentAudit'),(semantics,'semanticsRegistry'),(support,'candidateSupport')]: df.to_csv(paths[k],index=False)
exact=int(semantics.documentedExactMappingFound.sum())
summaryj={'schemaVersion':'pfi.p10-8.alkafri-mask-semantics-summary.v1','generatedAtUtc':datetime.now(timezone.utc).isoformat(),'sourceDatasetAudited':True,'manualAndProcessedMasksPresent':True,'trainingExecuted':False,'weightsDeserialized':False,'internalTestAccessed':False,'officialHiddenTestAccessed':False,'patientIdentifiersExported':False,'clinicalGroundTruthCreated':False,'clinicalThresholdsFrozen':False,'trainingAuthorized':False,'maskFileCount':int(len(priv)),'pngFileCount':int((priv.suffix=='.png').sum()),'xcfFileCount':int((priv.suffix=='.xcf').sum()),'sampledPngCount':int(len(profile)),'pairingKeyCount':int(len(pairreg)),'pixelComparisonCount':int(len(comparison)),'sampledXcfCount':int(len(xcfa)),'documentSourceCount':int(len(docaudit)),'exactDocumentedTokenMappings':exact,'unresolvedTokenCount':int((semantics.semanticStatus!='SEMANTICS_DOCUMENTED').sum()),'semanticMappingValidated':exact==3,'candidateFindingTrainingAuthorizedCount':0,'nextRequiredGate':'NOTEBOOK_73_VIABILITY_GATE'}
write_json(paths['summaryJson'],summaryj)
marker={'schemaVersion':'pfi.p10-8.notebook-72-complete.v1','status':'NOTEBOOK_72_COMPLETE',**summaryj,'outputs':{k:str(v) for k,v in paths.items()}}
write_json(OUT/'NOTEBOOK_72_COMPLETE.json',marker)
print(json.dumps(marker,indent=2,ensure_ascii=False)); print('NOTEBOOK_72_COMPLETE')


Máscaras/XCF: 30900 PNG: 23175 XCF: 7725


,sourceType,suffix,modality,dToken,nameParsed,fileCount,totalBytes,uniqueCaseCount
0,manual,.png,T1,D3,True,5150,6284001,515
1,manual,.png,T1,D4,True,5150,6423133,515
2,manual,.png,T1,D5,True,5150,6506475,515
3,manual,.xcf,T1,D3,True,2575,266512468,515
4,manual,.xcf,T1,D4,True,2575,266580562,515
5,manual,.xcf,T1,D5,True,2575,266495143,515
6,processed_intermediary,.png,T1,D3,True,515,28399959,515
7,processed_intermediary,.png,T1,D4,True,515,28740957,515
8,processed_intermediary,.png,T1,D5,True,515,29267947,515
9,processed_intermediary,.png,T2,D3,True,515,28380963,515


PNG perfilados: 120


,sourceType,modality,dToken,maskProfile,sampleCount
0,manual,T1,D3,MULTICOLOR,12
1,manual,T1,D4,MULTICOLOR,12
2,manual,T1,D5,MULTICOLOR,12
3,processed_intermediary,T1,D3,MULTICOLOR,12
4,processed_intermediary,T1,D4,MULTICOLOR,12
5,processed_intermediary,T1,D5,MULTICOLOR,12
6,processed_intermediary,T2,D3,MULTICOLOR,12
7,processed_intermediary,T2,D4,MULTICOLOR,12
8,processed_intermediary,T2,D5,MULTICOLOR,12
9,processed_intermediary,NaN,NaN,MULTICOLOR,7


Claves pairing: 3090 comparaciones: 120


,dToken,manualPngCount,processedPngCount,xcfCount,cooccurringDocumentationTerms,documentedExactMappingFound,semanticStatus,anatomicalMeaningValidated,clinicalFindingMeaningValidated,trainingAuthorized
0,D3,5150,1030,2575,disc|thecal_sac,False,SEMANTICS_INFERRED_REQUIRES_REVIEW,False,False,False
1,D4,5150,1030,2575,,False,UNKNOWN_NOT_USABLE,False,False,False
2,D5,5150,1030,2575,,False,UNKNOWN_NOT_USABLE,False,False,False


,findingType,documentationTermPresent,maskClassMappingValidated,caseSeriesLevelSliceAlignmentValidated,clinicalTaxonomyFrozen,supportStatus,trainingAuthorized
0,facet_hypertrophy,False,False,False,False,ANNOTATION_NOT_DEMONSTRATED,False
1,ligamentum_flavum_hypertrophy,False,False,False,False,ANNOTATION_NOT_DEMONSTRATED,False
2,annular_tear,False,False,False,False,ANNOTATION_NOT_DEMONSTRATED,False
3,nerve_root_compression,False,False,False,False,ANNOTATION_NOT_DEMONSTRATED,False
4,epidural_fat,False,False,False,False,ANNOTATION_NOT_DEMONSTRATED,False
5,disc_height,False,False,False,False,ANNOTATION_NOT_DEMONSTRATED,False
6,spondylolisthesis,False,False,False,False,ANNOTATION_NOT_DEMONSTRATED,False
7,disc_herniation,False,False,False,False,ANNOTATION_NOT_DEMONSTRATED,False
8,disc_bulging,False,False,False,False,ANNOTATION_NOT_DEMONSTRATED,False


{
  "schemaVersion": "pfi.p10-8.alkafri-mask-semantics-summary.v1",
  "status": "NOTEBOOK_72_COMPLETE",
  "generatedAtUtc": "2026-08-07T02:07:10.630588+00:00",
  "sourceDatasetAudited": true,
  "manualAndProcessedMasksPresent": true,
  "trainingExecuted": false,
  "weightsDeserialized": false,
  "internalTestAccessed": false,
  "officialHiddenTestAccessed": false,
  "patientIdentifiersExported": false,
  "clinicalGroundTruthCreated": false,
  "clinicalThresholdsFrozen": false,
  "trainingAuthorized": false,
  "maskFileCount": 30900,
  "pngFileCount": 23175,
  "xcfFileCount": 7725,
  "sampledPngCount": 120,
  "pairingKeyCount": 3090,
  "pixelComparisonCount": 120,
  "sampledXcfCount": 60,
  "documentSourceCount": 89,
  "exactDocumentedTokenMappings": 0,
  "unresolvedTokenCount": 3,
  "semanticMappingValidated": false,
  "candidateFindingTrainingAuthorizedCount": 0,
  "nextRequiredGate": "NOTEBOOK_73_VIABILITY_GATE",
  "outputs": {
    "maskInventory": "/content/drive/MyDrive/PFI_MVP/res

## Interpretación

PNG/XCF demuestra anotación gráfica, no una etiqueta degenerativa específica. Coincidencias documentales solo permiten `SEMANTICS_INFERRED_REQUIRES_REVIEW`. El Notebook 73 debe decidir viabilidad; este notebook mantiene `trainingAuthorized=false`.
